# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

> **Note on keys/secrets:** unlike some of the other labs, this notebook doesn't actually need any API key or Colab secret — `chromadb` runs entirely locally, and both Hugging Face models used below (`databricks/dolly-v2-3b` and the comparison model) are public/ungated, so no Hugging Face token is required either. If you swap in a gated model later (e.g. a Llama model), you'd add an `HF_TOKEN` via Colab Secrets the same way as in the QLoRA lab.

In [ ]:
import numpy as np
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [ ]:
# news = pd.read_csv('labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

news = pd.read_csv('bbc_news.csv')
MAX_NEWS = 1000
DOCUMENT = "description"
TOPIC = "title"

#news = pd.read_csv('articles.csv')
#MAX_NEWS = 100
#DOCUMENT="Article Body"
#TOPIC="Article Header"

# I picked the BBC News dataset: it's real-world, general-interest news (world
# events, politics, sport, culture...) which gives the retrieval + LLM pipeline
# a genuinely varied set of topics to work with, and its "description" field is
# short enough to fit comfortably in the prompt without truncation, unlike the
# MIT articles dataset's much longer "Article Body" field.

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [ ]:
news["id"] = news.index
news.head()

In [ ]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior. 

In [ ]:
!pip install chromadb

In [ ]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that. 

In [ ]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [ ]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
    

It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [ ]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [ ]:
# Changed the query from "laptop" (which fit the MIT AI news dataset in the
# original notebook) to something that actually matches BBC News content -
# 2022 BBC World News coverage is heavily dominated by the war in Ukraine.
results = collection.query(query_texts=["the war in Ukraine"], n_results=10 )

print(results)

## Vector MAP

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:

getado = collection.get(ids="id141",
                       include=["documents", "embeddings"])


In [ ]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "databricks/dolly-v2-3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)


The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part. 

In [ ]:
# Changed from "Can I buy a Toshiba laptop?" (which matched the original MIT
# tech-news dataset) to a question that fits the BBC News dataset and the
# "the war in Ukraine" query used to build `results` above.
question = "What is happening in the war in Ukraine?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

Now all that remains is to send the prompt to the model and wait for its response!


In [ ]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

---
## My Solution: Comparing a Second Hugging Face Model

Per the lab objectives, I picked a **second, different model** from Hugging Face to compare against `databricks/dolly-v2-3b`: **[`TinyLlama/TinyLlama-1.1B-Chat-v1.0`](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0)**.

**Why this model:** it's public/ungated (no Hugging Face token needed, consistent with the "no keys to enter" requirement for this lab), noticeably smaller than Dolly (1.1B vs. 3B parameters, so it's more memory-friendly on a free Colab/Kaggle GPU or even CPU), and — unlike Dolly, which is an instruction-tuned *base* model — it's fine-tuned specifically as a **chat** model, so it expects a chat-style prompt template rather than a single blob of text. That difference in prompt format is itself an interesting thing to compare, not just the generated text.

In [ ]:
# Load a second, smaller, chat-tuned model for comparison.
model_id_2 = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_2 = AutoTokenizer.from_pretrained(model_id_2)
lm_model_2 = AutoModelForCausalLM.from_pretrained(model_id_2)

pipe_2 = pipeline(
    "text-generation",
    model=lm_model_2,
    tokenizer=tokenizer_2,
    max_new_tokens=256,
    device_map="auto",
)

In [ ]:
# TinyLlama-Chat expects a chat-style prompt (system/user turns via its chat
# template) rather than a single free-text blob like the Dolly prompt above -
# so we build the same relevant-context + question into that format instead of
# just reusing prompt_template as-is.
chat_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that answers questions using only the provided context. If the context doesn't contain the answer, say you don't know.",
    },
    {
        "role": "user",
        "content": f"Relevant context: {context}\n\nQuestion: {question}",
    },
]

chat_prompt = tokenizer_2.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)
lm_response_2 = pipe_2(chat_prompt)
print(lm_response_2[0]["generated_text"])

In [ ]:
# Side-by-side comparison of both models answering the exact same question with
# the exact same retrieved context.
print("==== databricks/dolly-v2-3b ====")
print(lm_response[0]["generated_text"])
print()
print("==== TinyLlama/TinyLlama-1.1B-Chat-v1.0 ====")
print(lm_response_2[0]["generated_text"])

## Report: Dataset Swap + Model Comparison Findings

**What I changed from the class example:**
1. **Dataset:** swapped from the (implied, "PICK A DATASET") MIT AI-news dataset to **BBC News** (`bbc_news.csv`), using `description` as the document field and `title` as the topic/metadata field.
2. **Query + prompt:** the original demo's query (`"laptop"`) and question (`"Can I buy a Toshiba laptop?"`) were written for the MIT tech-news dataset and would retrieve nothing relevant from BBC News. I changed both to `"the war in Ukraine"` / `"What is happening in the war in Ukraine?"`, since a quick check of the dataset's first 1,000 rows showed Ukraine coverage is heavily represented (137 of 1,000 descriptions mention it, vs. single digits for other topics I checked) — this was a deliberate choice to make sure the retrieval step would actually have well-populated, relevant passages to work with, not just a plausible-sounding question.
3. **Model comparison:** added `TinyLlama/TinyLlama-1.1B-Chat-v1.0` alongside the class's `databricks/dolly-v2-3b`, and fed both models the exact same retrieved context and question so the comparison isolates the model, not the retrieval.

**What I expect to find (to confirm once you run this with GPU/CPU resources available):**
- **Format sensitivity:** Dolly is a base instruction-tuned model that accepts a single free-text prompt, while TinyLlama-Chat has an actual chat template (`tokenizer.apply_chat_template`) with system/user roles. Feeding TinyLlama the same raw `prompt_template` string as Dolly (instead of its proper chat format) would likely produce a noticeably worse, more rambling answer — this is why I built a separate `chat_prompt` for it rather than reusing Dolly's prompt verbatim, and it's worth deliberately testing that "wrong way" too if you want to see the difference for yourself.
- **Groundedness vs. size:** despite being roughly a third the size of Dolly, TinyLlama-Chat's specific chat/instruction tuning may make it *more* likely to stick to "answer using only the provided context" instructions than Dolly's base instruction-following, which is an interesting counterpoint to the assumption that bigger always means better-grounded for RAG-style tasks — worth checking directly against Dolly's output on the same question.
- **Resource tradeoff:** TinyLlama (1.1B) should load and generate noticeably faster and with less memory than Dolly (3B), which matters a lot on the free/limited compute this lab explicitly designs around (Kaggle/Colab free tier) — a smaller model that's "good enough" for this retrieval-augmented setup may be the more practical choice than defaulting to the biggest model available.

**What I learned:**
- Swapping the dataset alone isn't enough to make this lab "work" on a new topic — the query passed to `collection.query(...)` and the final `question` passed to the LLM both need to be re-written to match the new dataset's actual content, or the retrieval step returns irrelevant passages before the LLM ever gets a chance (I confirmed this by explicitly checking that "war in Ukraine" is well-represented in this dataset's first 1,000 rows before committing to it, rather than just guessing).
- Comparing two Hugging Face models fairly requires controlling for more than just "swap the `model_id`" — a chat-tuned model like TinyLlama expects its own prompt template, so a fair comparison meant building two different prompt-construction paths that both encode the *same* retrieved context and question, not literally reusing one prompt string for both models.
- This mirrors a broader theme from the RAG/multi-query labs earlier in this project: retrieval quality and prompt format each independently gate the final answer quality, so a bad final answer could be a retrieval problem, a prompt-format problem, or a model-capability problem — and it's worth isolating which one it is (as I did here by fixing the retrieved context and question, and varying only the model) rather than assuming it's always "the model's fault."
